# Arroyo Capital: inversión y portafolio en la industria del cine

**Proyecto final de Programación Analítica II**  
**Estudiante:** Vicente Alvarez  
**Universidad de Montevideo**

El objetivo es analizar la relación entre riesgo y retorno de distintos tipos de películas y construir una tesis de inversión para distribuir USD 500 millones.

## 1. Análisis exploratorio y correcciones de calidad

### 1.1 Carga inicial de los datos

Primero importamos las herramientas necesarias y definimos la ubicación de los archivos. Usamos rutas relativas para que el notebook funcione en otra computadora al clonar el repositorio.

In [1]:
from pathlib import Path
import pandas as pd

# Esta ruta funciona tanto si Jupyter inicia en la raíz como en notebooks/.
RAIZ_PROYECTO = Path.cwd()
if RAIZ_PROYECTO.name == "notebooks":
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent

RUTA_PELICULAS = RAIZ_PROYECTO / "datos" / "peliculas.csv"
RUTA_IPC = RAIZ_PROYECTO / "datos" / "ipc_eeuu.csv"

print(f"Películas: {RUTA_PELICULAS}")
print(f"IPC: {RUTA_IPC}")

Películas: C:\Users\Santi\Documents\Codex\programacion-analitica-2-portafolio-cine-alvarez\datos\peliculas.csv
IPC: C:\Users\Santi\Documents\Codex\programacion-analitica-2-portafolio-cine-alvarez\datos\ipc_eeuu.csv


Declaramos los tipos al cargar los archivos. Las variables de texto usan el tipo `string`, los enteros usan `Int64` y las variables continuas usan `Float64`. Estos tipos admiten valores faltantes, algo importante porque la consigna advierte que la base no está auditada. La fecha se convierte por separado con `errors="coerce"`, de modo que un valor inválido se transforma en faltante en vez de detener todo el notebook.

In [2]:
tipos_peliculas = {
    "id_pelicula": "Int64",
    "titulo": "string",
    "anio_estreno": "Int64",
    "fecha_estreno": "string",
    "genero": "string",
    "clasificacion": "string",
    "pais_origen": "string",
    "es_secuela": "string",
    "director": "string",
    "guionista": "string",
    "protagonista": "string",
    "productora": "string",
    "duracion_minutos": "Float64",
    "puntaje_imdb": "Float64",
    "cantidad_votos": "Float64",
    "presupuesto_usd": "Float64",
    "recaudacion_mundial_usd": "Float64",
}

peliculas = pd.read_csv(RUTA_PELICULAS, dtype=tipos_peliculas)
peliculas["fecha_estreno"] = pd.to_datetime(
    peliculas["fecha_estreno"], errors="coerce"
)

ipc = pd.read_csv(
    RUTA_IPC,
    dtype={"anio": "Int64", "ipc_eeuu": "Float64"},
)

Verificamos que ambos archivos se hayan cargado mostrando sus dimensiones, las primeras filas y los tipos resultantes. En esta instancia solo observamos: todavía no corregimos ni eliminamos datos.

In [3]:
print(f"peliculas.csv: {peliculas.shape[0]:,} filas y {peliculas.shape[1]} columnas")
print(f"ipc_eeuu.csv: {ipc.shape[0]:,} filas y {ipc.shape[1]} columnas")

display(peliculas.head())
display(ipc.head())

peliculas.csv: 7,740 filas y 17 columnas
ipc_eeuu.csv: 41 filas y 2 columnas


,id_pelicula,titulo,anio_estreno,fecha_estreno,genero,clasificacion,pais_origen,es_secuela,director,guionista,protagonista,productora,duracion_minutos,puntaje_imdb,cantidad_votos,presupuesto_usd,recaudacion_mundial_usd
0,15834,Hatchet II,2010,2010-07-01,Acción,Mayores de 17,Estados Unidos,Si,Adam Green,Adam Green,Danielle Harris,ArieScope Pictures,85.0,5.5,13000.0,800000.0,156190.0
1,10992,Shanghai Surprise,1986,1986-08-29,Aventura,Mayores de 13,UK,No,Jim Goddard,John Kohn,Sean Penn,HandMade Films,97.0,3.2,5000.0,17000000.0,2315683.0
2,15201,Mr. Magorium's Wonder Emporium,2007,2007-11-16,Comedia,Apta todo publico,Estados Unidos,No,Zach Helm,Zach Helm,Natalie Portman,Mandate Pictures,93.0,6.2,44000.0,65000000.0,69474661.0
3,13042,Celtic Pride,1996,1996-04-19,Comedia,Mayores de 13,Estados Unidos,No,Tom DeCerchio,Judd Apatow,Damon Wayans,Caravan Pictures,91.0,5.3,9600.0,<NA>,9255027.0
4,11961,Kafka,1991,1991-11-15,Drama,Mayores de 13,Francia,No,Steven Soderbergh,Lem Dobbs,Jeremy Irons,Baltimore Pictures,98.0,6.9,9600.0,11000000.0,1059071.0


,anio,ipc_eeuu
0,1980,82.4
1,1981,90.9
2,1982,96.5
3,1983,99.6
4,1984,103.9


In [4]:
print("Tipos de peliculas.csv:")
display(peliculas.dtypes.to_frame("tipo"))

print("Tipos de ipc_eeuu.csv:")
display(ipc.dtypes.to_frame("tipo"))

Tipos de peliculas.csv:


,tipo
id_pelicula,Int64
titulo,string
anio_estreno,Int64
fecha_estreno,datetime64[us]
genero,string
clasificacion,string
pais_origen,string
es_secuela,string
director,string
guionista,string


Tipos de ipc_eeuu.csv:


,tipo
anio,Int64
ipc_eeuu,Float64


**Conclusión de la carga:** los archivos quedaron disponibles para el análisis. `peliculas.csv` contiene 7.740 registros y 17 variables, mientras que `ipc_eeuu.csv` contiene 41 años y 2 variables. La carga conserva los faltantes y convierte las fechas inválidas en valores ausentes para auditarlas en los siguientes puntos.

### 1.2 Estructura y descripción general

En este punto resumimos la estructura de la base. Separamos las variables categóricas de las numéricas porque requieren medidas diferentes: en las primeras contamos categorías distintas y en las segundas observamos cantidad, promedio, dispersión, mínimos, cuartiles y máximos.

In [5]:
resumen_estructura = pd.DataFrame({
    "archivo": ["peliculas.csv", "ipc_eeuu.csv"],
    "filas": [peliculas.shape[0], ipc.shape[0]],
    "columnas": [peliculas.shape[1], ipc.shape[1]],
})
resumen_estructura

,archivo,filas,columnas
0,peliculas.csv,7740,17
1,ipc_eeuu.csv,41,2


In [6]:
columnas_categoricas = peliculas.select_dtypes(include="string").columns

categorias_por_variable = (
    peliculas[columnas_categoricas]
    .nunique(dropna=True)
    .sort_values()
    .rename("cantidad_categorias")
    .to_frame()
)
categorias_por_variable

,cantidad_categorias
es_secuela,2
clasificacion,10
genero,25
pais_origen,64
productora,2744
protagonista,2814
director,2949
guionista,4535
titulo,7512


In [7]:
columnas_numericas = peliculas.select_dtypes(include="number").columns

descripcion_numericas = (
    peliculas[columnas_numericas]
    .describe()
    .T
    .round(2)
)
descripcion_numericas

,count,mean,std,min,25%,50%,75%,max
id_pelicula,7740.0,13971.02,3920.34,10001.0,11921.75,13840.5,15759.25,90014.0
anio_estreno,7740.0,2000.4,11.15,1980.0,1991.0,2000.0,2010.0,2020.0
duracion_minutos,7736.0,108.83,44.86,0.0,95.0,104.0,116.0,999.0
puntaje_imdb,7737.0,6.37,1.03,0.0,5.8,6.5,7.1,9.3
cantidad_votos,7737.0,88122.35,163333.63,7.0,9100.0,33000.0,93000.0,2400000.0
presupuesto_usd,5551.0,35000792.4,41215579.81,-1.0,10000000.0,20000000.0,43000000.0,356000000.0
recaudacion_mundial_usd,7549.0,78600686.19,168497298.56,-622674139.0,4515258.0,20240502.0,76019048.0,2847246203.0


In [8]:
descripcion_ipc = ipc.describe().T.round(2)
descripcion_ipc

,count,mean,std,min,25%,50%,75%,max
anio,41.0,2000.0,11.98,1980.0,1990.0,2000.0,2010.0,2020.0
ipc_eeuu,41.0,173.46,52.66,82.4,130.7,172.2,218.1,258.8


**Conclusiones iniciales:**

- La base de películas tiene 7.740 filas y 17 columnas. La tabla del IPC cubre 41 años, desde 1980 hasta 2020.
- `clasificacion` presenta 10 categorías, aunque la documentación describe solo 6, y `genero` presenta 25. Esto anticipa categorías escritas de distintas maneras que deberán normalizarse.
- En votos, presupuesto y recaudación, el promedio es bastante mayor que la mediana. Esto indica distribuciones asimétricas, donde unas pocas películas muy grandes empujan el promedio hacia arriba.
- Aparecen mínimos o máximos imposibles: duración 0 y 999 minutos, puntaje IMDb 0, presupuesto -1 y recaudación negativa. Por ahora no se corrigen; quedan identificados para la auditoría de calidad.
- El presupuesto tiene menos observaciones válidas que las demás variables monetarias. El tratamiento de esos faltantes se decidirá más adelante después de estudiar si siguen algún patrón.

### 1.3 Duplicados exactos y películas repetidas

Una fila exactamente duplicada repite todos los valores y puede eliminarse sin perder información. En cambio, una película repetida con datos distintos representa un conflicto que debe investigarse antes de decidir qué registro conservar.

In [9]:
duplicados_exactos_extra = peliculas.duplicated().sum()
filas_en_duplicados_exactos = peliculas.duplicated(keep=False).sum()
porcentaje_duplicados_exactos = duplicados_exactos_extra / len(peliculas) * 100

resumen_duplicados = pd.DataFrame({
    "medida": [
        "copias exactas adicionales",
        "filas involucradas en duplicados exactos",
        "porcentaje de la base a eliminar",
    ],
    "valor": [
        duplicados_exactos_extra,
        filas_en_duplicados_exactos,
        round(porcentaje_duplicados_exactos, 2),
    ],
})
resumen_duplicados

,medida,valor
0,copias exactas adicionales,58.00
1,filas involucradas en duplicados exactos,116.00
2,porcentaje de la base a eliminar,0.75


In [10]:
ejemplos_duplicados_exactos = (
    peliculas[peliculas.duplicated(keep=False)]
    .sort_values(["id_pelicula", "titulo"])
    .head(10)
)
ejemplos_duplicados_exactos

,id_pelicula,titulo,anio_estreno,fecha_estreno,genero,clasificacion,pais_origen,es_secuela,director,guionista,protagonista,productora,duracion_minutos,puntaje_imdb,cantidad_votos,presupuesto_usd,recaudacion_mundial_usd
2780,10199,Scream,1981,1981-01-01,Terror,Mayores de 17,Estados Unidos,No,Byron Quisenberry,Byron Quisenberry,Pepper Martin,Calendar International Pictures,82.0,2.9,674.0,<NA>,<NA>
5907,10199,Scream,1981,1981-01-01,Terror,Mayores de 17,Estados Unidos,No,Byron Quisenberry,Byron Quisenberry,Pepper Martin,Calendar International Pictures,82.0,2.9,674.0,<NA>,<NA>
4905,10296,The Soldier,1982,1982-08-27,Acción,Mayores de 17,Estados Unidos,No,James Glickenhaus,James Glickenhaus,Ken Wahl,James Glickenhaus,96.0,5.3,1100.0,4000000.0,6328816.0
5448,10296,The Soldier,1982,1982-08-27,Acción,Mayores de 17,Estados Unidos,No,James Glickenhaus,James Glickenhaus,Ken Wahl,James Glickenhaus,96.0,5.3,1100.0,4000000.0,6328816.0
6992,10417,Escape from the Bronx,1983,1985-01-18,Acción,MAYORES DE 17,Italia,No,Enzo G. Castellari,Tito Carpi,Mark Gregory,FULVIA FILM,89.0,4.6,2300.0,<NA>,<NA>
7575,10417,Escape from the Bronx,1983,1985-01-18,Acción,MAYORES DE 17,Italia,No,Enzo G. Castellari,Tito Carpi,Mark Gregory,FULVIA FILM,89.0,4.6,2300.0,<NA>,<NA>
134,10499,Conan the Destroyer,1984,NaT,ACCION,Supervision parental,Estados Unidos,No,Richard Fleischer,Robert E. Howard,Arnold Schwarzenegger,Dino De Laurentiis Company,103.0,5.9,77000.0,18000000.0,31042035.0
6801,10499,Conan the Destroyer,1984,NaT,ACCION,Supervision parental,Estados Unidos,No,Richard Fleischer,Robert E. Howard,Arnold Schwarzenegger,Dino De Laurentiis Company,103.0,5.9,77000.0,18000000.0,31042035.0
4833,10645,The Goonies,1985,1985-06-07,Aventura,Supervisión parental,EE.UU.,No,Richard Donner,Chris Columbus,Sean Astin,Warner Bros.,114.0,7.8,252000.0,19000000.0,63980721.0
5321,10645,The Goonies,1985,1985-06-07,Aventura,Supervisión parental,EE.UU.,No,Richard Donner,Chris Columbus,Sean Astin,Warner Bros.,114.0,7.8,252000.0,19000000.0,63980721.0


Para buscar películas repetidas con datos distintos, primero quitamos temporalmente las copias exactas. Luego agrupamos por `titulo` y `anio_estreno`, una combinación razonable para identificar la misma película sin depender del identificador.

In [11]:
peliculas_sin_copias_exactas = peliculas.drop_duplicates()

mascara_repetidas_distintas = peliculas_sin_copias_exactas.duplicated(
    subset=["titulo", "anio_estreno"], keep=False
)
peliculas_repetidas_distintas = (
    peliculas_sin_copias_exactas[mascara_repetidas_distintas]
    .sort_values(["titulo", "anio_estreno", "id_pelicula"])
)

cantidad_grupos_repetidos = peliculas_repetidas_distintas.groupby(
    ["titulo", "anio_estreno"]
).ngroups

print(f"Películas repetidas con datos distintos: {cantidad_grupos_repetidos} pares")
print(f"Filas involucradas: {len(peliculas_repetidas_distintas)}")

Películas repetidas con datos distintos: 14 pares
Filas involucradas: 28


In [12]:
columnas_comparacion = [
    "id_pelicula",
    "titulo",
    "anio_estreno",
    "presupuesto_usd",
    "recaudacion_mundial_usd",
]
peliculas_repetidas_distintas[columnas_comparacion]

,id_pelicula,titulo,anio_estreno,presupuesto_usd,recaudacion_mundial_usd
1970,12898,Bio-Dome,1996,15000000.0,13427615.0
2653,90001,Bio-Dome,1996,19697118.0,13427615.0
2212,13101,Breakdown,1997,36000000.0,50159144.0
1747,90004,Breakdown,1997,31901667.0,50159144.0
5906,16378,Cloudy with a Chance of Meatballs 2,2013,78000000.0,274325949.0
3074,90008,Cloudy with a Chance of Meatballs 2,2013,93580153.0,274325949.0
6101,13883,Enemy at the Gates,2001,68000000.0,96976270.0
5510,90011,Enemy at the Gates,2001,43107007.0,96976270.0
5343,12897,Executive Decision,1996,55000000.0,121969216.0
6287,90002,Executive Decision,1996,59619028.0,121969216.0


**Conclusión sobre duplicados:** encontramos 58 copias exactas adicionales, equivalentes al 0,75 % de la base. También aparecen 14 pares de películas con el mismo título y año, pero con identificadores y presupuestos diferentes. En todos esos pares, uno de los identificadores comienza con 90000 y la recaudación coincide, mientras que cambia el presupuesto. Esto sugiere registros artificiales o alterados. Antes de modificar la base, dejamos documentada la decisión propuesta: eliminar las 58 copias exactas y, en los 14 pares conflictivos, conservar el registro con el identificador habitual y descartar el registro 90000. El efecto total sería retirar 72 filas, aproximadamente el 0,93 % de la base.

#### Corrección aplicada

Creamos `peliculas_limpias` para continuar el análisis sin alterar el DataFrame original. Primero retiramos las copias exactas y luego quitamos únicamente los 14 registros conflictivos cuyo identificador comienza con 90000.

In [13]:
filas_antes = len(peliculas)

# 1. Eliminamos solamente las copias adicionales de filas idénticas.
peliculas_limpias = peliculas.drop_duplicates().copy()

# 2. Identificamos los registros artificiales dentro de los pares conflictivos.
ids_conflictivos_90000 = peliculas_repetidas_distintas.loc[
    peliculas_repetidas_distintas["id_pelicula"] >= 90000,
    "id_pelicula",
].tolist()

peliculas_limpias = peliculas_limpias[
    ~peliculas_limpias["id_pelicula"].isin(ids_conflictivos_90000)
].copy()

filas_despues = len(peliculas_limpias)
filas_eliminadas = filas_antes - filas_despues
porcentaje_eliminado = filas_eliminadas / filas_antes * 100

In [14]:
control_correccion_duplicados = pd.DataFrame({
    "medida": [
        "filas antes",
        "copias exactas eliminadas",
        "registros 90000 eliminados",
        "filas después",
        "porcentaje total eliminado",
    ],
    "valor": [
        filas_antes,
        duplicados_exactos_extra,
        len(ids_conflictivos_90000),
        filas_despues,
        round(porcentaje_eliminado, 2),
    ],
})
control_correccion_duplicados

,medida,valor
0,filas antes,7740.00
1,copias exactas eliminadas,58.00
2,registros 90000 eliminados,14.00
3,filas después,7668.00
4,porcentaje total eliminado,0.93


In [15]:
# Comprobaciones posteriores a la corrección.
assert peliculas_limpias.duplicated().sum() == 0
assert not peliculas_limpias["id_pelicula"].isin(ids_conflictivos_90000).any()

print("No quedan duplicados exactos ni identificadores conflictivos 90000.")

No quedan duplicados exactos ni identificadores conflictivos 90000.


**Resultado de la corrección:** la base de trabajo pasa de 7.740 a 7.668 filas. Se eliminaron 72 registros, equivalentes al 0,93 % del total. El CSV original y el DataFrame `peliculas` permanecen sin cambios, por lo que la transformación es reproducible y auditable. Desde este punto, los análisis siguientes utilizarán `peliculas_limpias`.

### 1.4 Valores faltantes

Cuantificamos los faltantes sobre `peliculas_limpias`. Reportamos tanto la cantidad como el porcentaje porque una misma cantidad puede tener distinta importancia según el tamaño de la base. En esta etapa no imputamos ni eliminamos datos.

In [16]:
resumen_faltantes = pd.DataFrame({
    "cantidad_faltantes": peliculas_limpias.isna().sum(),
    "porcentaje_faltantes": (peliculas_limpias.isna().mean() * 100).round(2),
})

resumen_faltantes = resumen_faltantes.sort_values(
    "cantidad_faltantes", ascending=False
)
resumen_faltantes

,cantidad_faltantes,porcentaje_faltantes
presupuesto_usd,2171,28.31
fecha_estreno,1167,15.22
recaudacion_mundial_usd,189,2.46
clasificacion,77,1.00
productora,15,0.20
duracion_minutos,4,0.05
puntaje_imdb,3,0.04
pais_origen,3,0.04
cantidad_votos,3,0.04
guionista,3,0.04


In [17]:
filas_con_algun_faltante = peliculas_limpias.isna().any(axis=1).sum()
porcentaje_filas_con_faltantes = (
    filas_con_algun_faltante / len(peliculas_limpias) * 100
)

print(f"Filas con al menos un faltante: {filas_con_algun_faltante:,}")
print(f"Porcentaje de filas afectadas: {porcentaje_filas_con_faltantes:.2f}%")

Filas con al menos un faltante: 3,074
Porcentaje de filas afectadas: 40.09%


El siguiente filtro muestra ejemplos de filas afectadas. Incluimos las variables principales para observar que los faltantes no siempre aparecen en las mismas columnas.

In [18]:
columnas_para_ejemplos = [
    "id_pelicula",
    "titulo",
    "anio_estreno",
    "fecha_estreno",
    "clasificacion",
    "duracion_minutos",
    "puntaje_imdb",
    "presupuesto_usd",
    "recaudacion_mundial_usd",
]

ejemplos_filas_con_faltantes = peliculas_limpias.loc[
    peliculas_limpias.isna().any(axis=1), columnas_para_ejemplos
].head(10)

ejemplos_filas_con_faltantes

,id_pelicula,titulo,anio_estreno,fecha_estreno,clasificacion,duracion_minutos,puntaje_imdb,presupuesto_usd,recaudacion_mundial_usd
3,13042,Celtic Pride,1996,1996-04-19,Mayores de 13,91.0,5.3,<NA>,9255027.0
5,11435,"Distant Voices, Still Lives",1988,NaT,Mayores de 13,84.0,7.4,<NA>,771745.0
7,10739,The Holcroft Covenant,1985,1985-10-18,Mayores de 17,112.0,5.7,<NA>,393825.0
8,12429,Silent Tongue,1993,NaT,Mayores de 13,102.0,5.3,<NA>,61274.0
10,15664,The American,2010,NaT,Mayores de 17,105.0,6.3,20000000.0,67876281.0
11,11791,An Angel at My Table,1990,1990-09-20,MAYORES DE 17,158.0,7.5,<NA>,1055995.0
12,11681,Nikita,1990,1990-07-01,Mayores de 17,117.0,7.3,<NA>,5017971.0
13,16780,Into the Forest,2015,2016-07-29,Mayores de 17,101.0,5.8,<NA>,92166.0
15,14557,District B13,2004,2006-06-02,Mayores de 17,84.0,7.1,<NA>,11169386.0
17,10240,Six Pack,1982,1982-07-16,Supervision parental,108.0,5.9,<NA>,20225989.0


**Conclusiones sobre faltantes:** el presupuesto es la variable más afectada, con 2.171 valores ausentes (28,31 %), por lo que eliminar todas esas filas sin estudiarlas podría sesgar el análisis. La fecha de estreno tiene 1.167 faltantes (15,22 %) y la recaudación mundial 189 (2,46 %). En total, 3.074 películas (40,09 %) tienen al menos un dato faltante, aunque muchas conservan la información necesaria para otros análisis. Por este motivo no aplicamos una eliminación general. El tratamiento del presupuesto se decidirá después de comparar sus faltantes por década, país y género, tal como exige la consigna.

### 1.5 Valores imposibles y centinelas

Buscamos valores que contradicen la definición de cada variable. También distinguimos un error evidente de un valor extremo posible: por ejemplo, una película puede durar más de 300 minutos, pero no puede durar 0 minutos ni tener una duración de 999 usada como centinela.

In [19]:
condiciones_imposibles = {
    "año fuera de 1980-2020": ~peliculas_limpias["anio_estreno"].between(1980, 2020),
    "duración igual a 0": peliculas_limpias["duracion_minutos"] == 0,
    "duración centinela 999": peliculas_limpias["duracion_minutos"] == 999,
    "IMDb fuera de 1-10": (
        ~peliculas_limpias["puntaje_imdb"].between(1, 10)
        & peliculas_limpias["puntaje_imdb"].notna()
    ),
    "votos negativos": peliculas_limpias["cantidad_votos"] < 0,
    "presupuesto menor o igual a 0": peliculas_limpias["presupuesto_usd"] <= 0,
    "recaudación negativa": peliculas_limpias["recaudacion_mundial_usd"] < 0,
}

resumen_valores_imposibles = pd.DataFrame({
    "problema": condiciones_imposibles.keys(),
    "filas_afectadas": [mascara.sum() for mascara in condiciones_imposibles.values()],
})
resumen_valores_imposibles["porcentaje_base"] = (
    resumen_valores_imposibles["filas_afectadas"]
    / len(peliculas_limpias)
    * 100
).round(2)
resumen_valores_imposibles

,problema,filas_afectadas,porcentaje_base
0,año fuera de 1980-2020,0,0.00
1,duración igual a 0,18,0.23
2,duración centinela 999,16,0.21
3,IMDb fuera de 1-10,22,0.29
4,votos negativos,0,0.00
5,presupuesto menor o igual a 0,48,0.63
6,recaudación negativa,11,0.14


In [20]:
columnas_revision = [
    "id_pelicula",
    "titulo",
    "anio_estreno",
    "duracion_minutos",
    "puntaje_imdb",
    "presupuesto_usd",
    "recaudacion_mundial_usd",
]

mascara_revision = pd.concat(condiciones_imposibles, axis=1).any(axis=1)
ejemplos_valores_imposibles = peliculas_limpias.loc[
    mascara_revision, columnas_revision
].head(20)
ejemplos_valores_imposibles

,id_pelicula,titulo,anio_estreno,duracion_minutos,puntaje_imdb,presupuesto_usd,recaudacion_mundial_usd
50,17258,Ant-Man and the Wasp,2018,118.0,7.0,162000000.0,-622674139.0
60,10819,Bliss,1985,999.0,6.9,<NA>,<NA>
117,16808,Rock the Kasbah,2015,106.0,5.5,-1.0,3394174.0
249,14789,Hustle & Flow,2005,116.0,7.3,0.0,23563727.0
305,16426,The Past,2013,130.0,0.0,11000000.0,12673462.0
356,15751,Dear John,2010,108.0,6.3,0.0,114984666.0
373,12914,Beavis and Butt-Head Do America,1996,81.0,6.8,-1.0,63118386.0
377,14167,Rollerball,2002,98.0,0.0,70000000.0,25852764.0
384,12092,Lethal Weapon 3,1992,118.0,6.7,-1.0,321731527.0
524,11337,Monkey Shines,1988,113.0,6.2,-1.0,5344577.0


In [21]:
print("Valores problemáticos de duración:")
display(
    peliculas_limpias.loc[
        peliculas_limpias["duracion_minutos"].isin([0, 999]),
        "duracion_minutos",
    ].value_counts().rename("cantidad").to_frame()
)

print("Valores no positivos de presupuesto:")
display(
    peliculas_limpias.loc[
        peliculas_limpias["presupuesto_usd"] <= 0,
        "presupuesto_usd",
    ].value_counts().rename("cantidad").to_frame()
)

Valores problemáticos de duración:


,cantidad
duracion_minutos,
0.0,18
999.0,16


Valores no positivos de presupuesto:


,cantidad
presupuesto_usd,
0.0,26
-1.0,22


**Conclusiones y corrección propuesta:**

- Hay 18 duraciones iguales a 0 y 16 iguales a 999. Ambos valores funcionan como faltantes disfrazados y deberían convertirse en `NaN`. Las duraciones reales de 357 y 366 minutos se conservan porque pertenecen a películas excepcionalmente largas, pero posibles.
- Hay 22 puntajes IMDb iguales a 0. Como la escala válida comienza en 1, deberían convertirse en `NaN`.
- Hay 48 presupuestos no positivos: 26 ceros y 22 valores `-1`. No representan presupuestos reales y deberían convertirse en `NaN`.
- Hay 11 recaudaciones negativas. La recaudación bruta no puede ser negativa y las magnitudes coinciden con valores plausibles escritos con signo incorrecto, por lo que se propone corregirlas tomando su valor absoluto.
- No encontramos años fuera de 1980-2020 ni cantidades de votos negativas.

Todavía no aplicamos estas correcciones: primero dejamos documentada la regla para aprobarla antes de modificar la base de trabajo.

#### Corrección aplicada

Aplicamos las reglas aprobadas sobre `peliculas_limpias`. Los centinelas se convierten en faltantes reales para que Pandas no los incluya en promedios o medianas. Las recaudaciones negativas se corrigen con valor absoluto, ya que el problema es únicamente el signo.

In [22]:
mascara_duracion_invalida = peliculas_limpias["duracion_minutos"].isin([0, 999])
mascara_imdb_invalido = peliculas_limpias["puntaje_imdb"] == 0
mascara_presupuesto_invalido = peliculas_limpias["presupuesto_usd"] <= 0
mascara_recaudacion_negativa = peliculas_limpias["recaudacion_mundial_usd"] < 0

mascara_alguna_correccion = (
    mascara_duracion_invalida
    | mascara_imdb_invalido
    | mascara_presupuesto_invalido
    | mascara_recaudacion_negativa
)

faltantes_antes_correccion = peliculas_limpias[[
    "duracion_minutos",
    "puntaje_imdb",
    "presupuesto_usd",
    "recaudacion_mundial_usd",
]].isna().sum()

In [23]:
peliculas_limpias.loc[mascara_duracion_invalida, "duracion_minutos"] = pd.NA
peliculas_limpias.loc[mascara_imdb_invalido, "puntaje_imdb"] = pd.NA
peliculas_limpias.loc[mascara_presupuesto_invalido, "presupuesto_usd"] = pd.NA
peliculas_limpias.loc[
    mascara_recaudacion_negativa, "recaudacion_mundial_usd"
] = peliculas_limpias.loc[
    mascara_recaudacion_negativa, "recaudacion_mundial_usd"
].abs()

In [24]:
faltantes_despues_correccion = peliculas_limpias[[
    "duracion_minutos",
    "puntaje_imdb",
    "presupuesto_usd",
    "recaudacion_mundial_usd",
]].isna().sum()

control_correcciones = pd.DataFrame({
    "faltantes_antes": faltantes_antes_correccion,
    "faltantes_despues": faltantes_despues_correccion,
    "nuevos_faltantes_identificados": (
        faltantes_despues_correccion - faltantes_antes_correccion
    ),
})
control_correcciones

,faltantes_antes,faltantes_despues,nuevos_faltantes_identificados
duracion_minutos,4,38,34
puntaje_imdb,3,25,22
presupuesto_usd,2171,2219,48
recaudacion_mundial_usd,189,189,0


In [25]:
filas_corregidas = mascara_alguna_correccion.sum()
porcentaje_filas_corregidas = filas_corregidas / len(peliculas_limpias) * 100

assert not peliculas_limpias["duracion_minutos"].isin([0, 999]).any()
assert not (peliculas_limpias["puntaje_imdb"] == 0).any()
assert not (peliculas_limpias["presupuesto_usd"] <= 0).any()
assert not (peliculas_limpias["recaudacion_mundial_usd"] < 0).any()

print(f"Filas afectadas por alguna corrección: {filas_corregidas}")
print(f"Porcentaje de la base afectado: {porcentaje_filas_corregidas:.2f}%")
print("Las comprobaciones posteriores no encontraron valores imposibles pendientes.")

Filas afectadas por alguna corrección: 115
Porcentaje de la base afectado: 1.50%
Las comprobaciones posteriores no encontraron valores imposibles pendientes.


**Resultado de la corrección:** se modificaron 115 filas, equivalentes al 1,50 % de la base limpia. Los faltantes de duración aumentan de 4 a 38, los de IMDb de 3 a 25 y los de presupuesto de 2.171 a 2.219. La recaudación conserva 189 faltantes porque sus 11 valores negativos fueron corregidos, no eliminados. Las comprobaciones confirman que ya no permanecen los centinelas tratados.